In [ ]:
# from dotenv import load_dotenv
# import os 
from google import genai

# load_dotenv()

client = genai.Client()

response = client.models.generate_content(
    model = "gemini-2.5-flash", contents="Tell me a programming joke"
)

response

GenerateContentResponse(
  automatic_function_calling_history=[],
  candidates=[
    Candidate(
      content=Content(
        parts=[
          Part(
            text="""Why do programmers prefer dark mode?

Because bugs are attracted to light!"""
          ),
        ],
        role='model'
      ),
      finish_reason=<FinishReason.STOP: 'STOP'>,
      index=0
    ),
  ],
  model_version='gemini-2.5-flash',
  response_id='aZq-aPKyCMTWvdIPkP7agA4',
  sdk_http_response=HttpResponse(
    headers=<dict len=11>
  ),
  usage_metadata=GenerateContentResponseUsageMetadata(
    candidates_token_count=15,
    prompt_token_count=6,
    prompt_tokens_details=[
      ModalityTokenCount(
        modality=<MediaModality.TEXT: 'TEXT'>,
        token_count=6
      ),
    ],
    thoughts_token_count=552,
    total_token_count=573
  )
)

In [5]:
def ask_llm(prompt):
    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt
    )

    return response.text

ask_llm("Du är en Göteborgare, ge mig ett skämt som är go")

'Hallå där, kompis! Här kommer ett skämt som e så gött att du får ett litet flin på läpparna, la.\n\nDet va två gubbar som satt å gôbba på en bänk vid Kungsportsplatsen, ju.\nDå säger den ena: "Asså, jag åt en sån gôrfärsk makrill igår kväll. Den va sååå go!"\nDå tittar den andre på\'n å säger: "Va? Hade du inte rensat den än, la?"\n\nÄr den inte gôrrrolig, la? Ha de gött!'

In [6]:
response = ask_llm("""
    Du är en expert inom köp och sälj av bostäder, likt en proffsig mäklare.
    Generera bostadspriser, månadsavgifter, address, stad, boarea i jsonformat (ej markdown)

    Exempel:
            {
                "address": "Fågelvägen 5,
                "price_sek": 3000000,
                "city": "Göteborg",
                "monthly_fee": 4000,
                "area": 60
            }   
                   
    Ge mig en lista på 5 bostäder
""")

response

'[\n    {\n        "address": "Drottninggatan 100B",\n        "price_sek": 6500000,\n        "city": "Stockholm",\n        "monthly_fee": 4500,\n        "area": 75\n    },\n    {\n        "address": "Kastanjegatan 22",\n        "price_sek": 4800000,\n        "city": "Malmö",\n        "monthly_fee": 3500,\n        "area": 140\n    },\n    {\n        "address": "Linnégatan 50A",\n        "price_sek": 4200000,\n        "city": "Göteborg",\n        "monthly_fee": 5200,\n        "area": 90\n    },\n    {\n        "address": "Studentgatan 8B",\n        "price_sek": 1950000,\n        "city": "Uppsala",\n        "monthly_fee": 2500,\n        "area": 28\n    },\n    {\n        "address": "Björkgatan 15",\n        "price_sek": 3900000,\n        "city": "Västerås",\n        "monthly_fee": 4000,\n        "area": 160\n    }\n]'

In [7]:
print(response)

[
    {
        "address": "Drottninggatan 100B",
        "price_sek": 6500000,
        "city": "Stockholm",
        "monthly_fee": 4500,
        "area": 75
    },
    {
        "address": "Kastanjegatan 22",
        "price_sek": 4800000,
        "city": "Malmö",
        "monthly_fee": 3500,
        "area": 140
    },
    {
        "address": "Linnégatan 50A",
        "price_sek": 4200000,
        "city": "Göteborg",
        "monthly_fee": 5200,
        "area": 90
    },
    {
        "address": "Studentgatan 8B",
        "price_sek": 1950000,
        "city": "Uppsala",
        "monthly_fee": 2500,
        "area": 28
    },
    {
        "address": "Björkgatan 15",
        "price_sek": 3900000,
        "city": "Västerås",
        "monthly_fee": 4000,
        "area": 160
    }
]


In [8]:
from pydantic import BaseModel, Field
import json 

class Apartment(BaseModel):
    address: str 
    city: str 
    price_sek: int = Field(gt=1000000, lt = 8000000) 
    monthly_fee: int 
    area: int 

class ApartmentList(BaseModel):
    objects: list[Apartment]


apartments = ApartmentList.model_validate({"objects": json.loads(response)})
apartments

ApartmentList(objects=[Apartment(address='Drottninggatan 100B', city='Stockholm', price_sek=6500000, monthly_fee=4500, area=75), Apartment(address='Kastanjegatan 22', city='Malmö', price_sek=4800000, monthly_fee=3500, area=140), Apartment(address='Linnégatan 50A', city='Göteborg', price_sek=4200000, monthly_fee=5200, area=90), Apartment(address='Studentgatan 8B', city='Uppsala', price_sek=1950000, monthly_fee=2500, area=28), Apartment(address='Björkgatan 15', city='Västerås', price_sek=3900000, monthly_fee=4000, area=160)])

In [9]:
apartments.objects

[Apartment(address='Drottninggatan 100B', city='Stockholm', price_sek=6500000, monthly_fee=4500, area=75),
 Apartment(address='Kastanjegatan 22', city='Malmö', price_sek=4800000, monthly_fee=3500, area=140),
 Apartment(address='Linnégatan 50A', city='Göteborg', price_sek=4200000, monthly_fee=5200, area=90),
 Apartment(address='Studentgatan 8B', city='Uppsala', price_sek=1950000, monthly_fee=2500, area=28),
 Apartment(address='Björkgatan 15', city='Västerås', price_sek=3900000, monthly_fee=4000, area=160)]

In [10]:
apartments.objects[1].address, apartments.objects[1].city

('Kastanjegatan 22', 'Malmö')

In [11]:
addresses = [apartment.address for apartment in apartments.objects ]

addresses

['Drottninggatan 100B',
 'Kastanjegatan 22',
 'Linnégatan 50A',
 'Studentgatan 8B',
 'Björkgatan 15']

In [12]:
addresses = [
    apartment.address
    for apartment in apartments.objects
    if apartment.price_sek < 4000000
]

addresses

['Studentgatan 8B', 'Björkgatan 15']

Get address, city, price, monthly_fee for the interval 4M - 8M

ladda in datan till en duckdb databas